In [0]:
! databricks secrets create-scope --scope jdbc-secrets

In [0]:
! databricks secrets put --scope jdbc-secrets --key db-host --string-value "adventureworks.postgres.database.azure.com"
! databricks secrets put --scope jdbc-secrets --key db-port --string-value "5432"
! databricks secrets put --scope jdbc-secrets --key db-database --string-value "adventureworks"
! databricks secrets put --scope jdbc-secrets --key db-user --string-value "readonly"
! databricks secrets put --scope jdbc-secrets --key db-password --string-value "Atlas@123"

In [0]:
# Install JDBC DDriver: Ensure your Databricks cluster has the PostgreSQL JDBC driver installed. You can add it via the Cluster UI -> Libraries -> Coordinates (org.postgresql:postgresql:42.6.0)

In [0]:
import pyspark.sql.functions as F

# --- Connection Configuration ---
jdbc_hostname = dbutils.secrets.get(scope="jdbc-secrets", key="db-host")
jdbc_port = dbutils.secrets.get(scope="jdbc-secrets", key="db-port")
jdbc_database = dbutils.secrets.get(scope="jdbc-secrets", key="db-database")
jdbc_user = dbutils.secrets.get(scope="jdbc-secrets", key="db-user")
jdbc_password = dbutils.secrets.get(scope="jdbc-secrets", key="db-password")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
  "user": jdbc_user,
  "password": jdbc_password,
  "driver": "org.postgresql.Driver"
}

# --- Data Loading ---
# Example: Load SalesOrderHeader table
try:
    # Adjust schema and table name as needed
    sales_order_header_df = spark.read.jdbc(
        url=jdbc_url,
        table="Sales.SalesOrderHeader", # Use actual schema.table
        properties=connection_properties
    )
    print("Successfully loaded data from Sales.SalesOrderHeader")
    display(sales_order_header_df.limit(5))

except Exception as e:
    print(f"Error connecting to or reading from database: {e}")
    dbutils.notebook.exit("Database connection failed")


In [0]:
from pyspark.sql.functions import col
# --- Basic Preparation & Selection ---
# Select relevant columns and handle potential data issues (e.g., nulls)
# For simplicity, let's select a few columns and drop rows with null target
# WARNING: This is a very basic example. Real-world prep is more complex.
selected_cols = ["SalesOrderID", "OrderDate", "CustomerID", "SubTotal", "TaxAmt", "Freight", "TotalDue"]
prepared_df = sales_order_header_df.select(*selected_cols).na.drop(subset=["TotalDue"])

# Add a primary key column if not obvious (needed for Feature Store)
# SalesOrderID is likely the primary key here.
prepared_df = prepared_df.withColumn("primary_key", F.col("SalesOrderID"))

prepared_df = prepared_df.withColumn("SubTotal", col("SubTotal").cast("float")) \
                 .withColumn("TaxAmt", col("TaxAmt").cast("float")) \
                 .withColumn("Freight", col("Freight").cast("float")) \
                     .withColumn("TotalDue", col("TotalDue").cast("float")) \

# Persist intermediate results (optional, but good for checkpoints)
prepared_df.write.format("delta").mode("overwrite").save("/mnt/adventureworks/prepared_data2")
# prepared_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data")

print("Data preparation basic steps completed.")
display(prepared_df.limit(5))

# Pass the prepared DataFrame path or name to the next notebook if needed
# For simplicity, we'll assume subsequent notebooks re-run this or load the saved Delta table.
dbutils.notebook.exit(prepared_df.count()) # Example exit value